- title: Занятие 6
- author: Sergey Korpachev
- date: 2026-10-06
- slug: 2026_lpr_lab06
- lecture_comment: Лекция 6
- lecture_link: lectures/2026_lpr_lec06.html
- lecture_link_ipynb: download/2026_lpr_lec06.ipynb
- ipynb_url: download/2026_lpr_lab06.ipynb

# Лабораторное занятие 6

# Дополнительные темы по ООП. Методы классов.

На прошлом занятии мы разобрали основы ООП: классы, объекты, инкапсуляцию, наследование и полиморфизм. Сегодня углубимся в тему методов, посмотрим на декораторы `@staticmethod`, `@classmethod` и `@property`, познакомимся с `dataclass`, а также разберём магические методы.

## 1. Краткое повторение: типы методов.

В Python методы делятся на три вида, в зависимости от того, что они получают первым аргументом:

| Тип метода | Первый аргумент | Декоратор | Когда использовать |
|---|---|---|---|
| Обычный (instance method) | `self` | нет | Работа с конкретным объектом |
| Статический (static method) | нет | `@staticmethod` | Утилитарная функция, логически связанная с классом |
| Классовый (class method) | `cls` | `@classmethod` | Работа с классом целиком, альтернативные конструкторы |

Разберём каждый подробнее.

## 2. Обычные методы.

Обычный метод получает первым аргументом `self` - ссылку на конкретный экземпляр класса. Через `self` метод может читать и изменять атрибуты объекта.

In [ ]:
class Counter:
    def __init__(self):
        self.value = 0

    def increment(self):
        self.value += 1
        return self.value

c = Counter()
print(c.increment())
print(c.increment())
print(c.value)

1
2
2


In [ ]:
class Counter:
    def __init__(self):
        self.value = 0
        print('1: ', self)

    def increment(self):
        self.value += 1
        return self.value

c = Counter()
print('2: ', c)

1:  <__main__.Counter object at 0x78f98ec92270>
2:  <__main__.Counter object at 0x78f98ec92270>


## 3. Статические методы (@staticmethod).

Статический метод не получает ни `self`, ни `cls`. По сути это обычная функция, которая лежит внутри класса - для логической группировки. Вызывать её можно и через класс, и через экземпляр.

Когда использовать:

- функция логически относится к классу, но не работает с его данными;
- утилитарные вычисления, валидация, преобразования.

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @staticmethod
    def celsius_to_fahrenheit(celsius):
        return celsius * 9 / 5 + 32

    @staticmethod
    def is_freezing(celsius):
        return celsius <= 0

# вызываем через класс
print(Temperature.celsius_to_fahrenheit(25))
print(Temperature.is_freezing(-5))

# или через экземпляр - тоже работает
t = Temperature(10)
print(t.celsius_to_fahrenheit(0))

77.0
True
32.0


## 4. Классовые методы (@classmethod).

Классовый метод получает первым аргументом `cls` - ссылку на сам класс, а не на экземпляр. Это позволяет:

- создавать альтернативные конструкторы (фабричные методы);
- работать с атрибутами класса;
- учитывать наследование: если вызвать метод у подкласса, `cls` будет этим подклассом.

In [ ]:
import datetime

class Person:
    species = "Homo sapiens"

    def __init__(self, name, birth_year):
        self.name = name
        self.birth_year = birth_year

    @classmethod
    def from_age(cls, name, age):
        """Альтернативный конструктор: создаёт Person по возрасту."""
        current_year = datetime.date.today().year
        return cls(name, current_year - age)

    @classmethod
    def get_species(cls):
        return cls.species

    def info(self):
        return f"{self.name}, родился в {self.birth_year}"

p1 = Person("Иван", 1990)
p2 = Person.from_age("Мария", 25)

print(p1.info())
print(p2.info())
print(Person.get_species())

Иван, родился в 1990
Мария, родился в 2001
Homo sapiens


### 4.1. Наследование и classmethod.

Классовый метод учитывает, у какого класса он вызван. Это удобно для фабрик.

In [ ]:
class Employee(Person):
    species = "Homo sapiens (работник)"

    def __init__(self, name, birth_year, position):
        super().__init__(name, birth_year)
        self.position = position

    @classmethod
    def from_age(cls, name, age):
        # используем ту же логику, но возвращаем Employee
        current_year = datetime.date.today().year
        return cls(name, current_year - age, "стажёр")

e = Employee.from_age("Пётр", 30)
print(e.info())
print(e.position)
print(Employee.get_species())  # переопределённый species

Пётр, родился в 1996
стажёр
Homo sapiens (работник)


## 5. Свойства (@property).

Иногда хочется обращаться к вычисляемому значению как к атрибуту, а не вызывать метод со скобками. Для этого есть декоратор `@property`.

In [ ]:
class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def area(self):
        return 3.14159 * self.radius ** 2

    @property
    def diameter(self):
        return self.radius * 2

c = Circle(5)
print(c.area)  # без скобок - как атрибут
print(c.diameter)

78.53975
10


Можно сделать и сеттер - чтобы контролировать изменение значения:

In [ ]:
class Temperature2:
    def __init__(self, celsius):
        self._celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("Температура не может быть ниже абсолютного нуля")
        self._celsius = value

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32

t = Temperature2(25)
print(t.fahrenheit)

t.celsius = 100
print(t.fahrenheit)

try:
    t.celsius = -300
except ValueError as err:
    print("Ошибка:", err)

77.0
212.0
Ошибка: Температура не может быть ниже абсолютного нуля


## 6. Дополнение к инкапсуляции: соглашения об уровне доступа.


В Python нет настоящих `private` / `protected`, как в C++ или Java. Вместо этого есть соглашения:

- `name` - публичный атрибут.
- `_name` - "защищённый" (protected). Не трогайте снаружи без необходимости.
- `__name` - "приватный" (name mangling). Python переименует его в `_ClassName__name`, что мешает случайному доступу.

In [ ]:
class Account:
    def __init__(self, owner, balance):
        self.owner = owner        # публичный
        self._currency = "RUB"    # защищённый
        self.__balance = balance  # приватный

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Сумма должна быть положительной")
        self.__balance += amount

    def get_balance(self):
        return self.__balance

acc = Account("Иван", 1000)
print(acc.owner)
print(acc._currency)  # работает, но так делать не рекомендуется

try:
    print(acc.__balance)
except AttributeError as err:
    print("Ошибка:", err)

# а вот так - сработает (name mangling)
print(acc._Account__balance)

Иван
RUB
Ошибка: 'Account' object has no attribute '__balance'
1000


## 7. Дополнение к наследованию: super() и MRO.

`super()` позволяет обратиться к методам родительского класса. В Python есть MRO (Method Resolution Order) - порядок, в котором ищутся методы.

In [ ]:
class A:
    def who(self):
        print("A")

class B(A):
    def who(self):
        print("B")
        super().who()

class C(A):
    def who(self):
        print("C")
        super().who()

class D(B, C):
    def who(self):
        print("D")
        super().who()

D().who()
print(D.__mro__)

D
B
C
A
(<class '__main__.D'>, <class '__main__.B'>, <class '__main__.C'>, <class '__main__.A'>, <class 'object'>)


## 8. Дополнение к полиморфизму: утиная типизация.

Python придерживается принципа "если это выглядит как утка и крякает как утка - это утка". Наследование не обязательно, если объект поддерживает нужные методы.

In [ ]:
class Duck:
    def quack(self):
        print("Кря!")

class Person:
    def quack(self):
        print("Человек подражает утке: кря!")

def make_it_quack(obj):
    obj.quack()

make_it_quack(Duck())
make_it_quack(Person())

Кря!
Человек подражает утке: кря!


## 9. Абстракция (кратко).

Абстрактный класс - это класс, который нельзя инстанцировать. Он задаёт интерфейс для наследников. В Python для этого используют модуль `abc`.

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self):
        pass

    @abstractmethod
    def perimeter(self):
        pass

class Rectangle(Shape):
    def __init__(self, w, h):
        self.w = w
        self.h = h

    def area(self):
        return self.w * self.h

    def perimeter(self):
        return 2 * (self.w + self.h)

# Shape()  # нельзя - вызовет TypeError
r = Rectangle(3, 4)
print(r.area(), r.perimeter())

12 14


Абстракция полезна, когда вы хотите заставить наследников реализовать определённые методы.

## 10. dataclass.

`dataclass` - декоратор из модуля `dataclasses`, который автоматически генерирует `__init__`, `__repr__` и `__eq__`. Идеально для классов, которые в основном хранят данные.

In [ ]:
from dataclasses import dataclass

@dataclass
class Point:
    x: float
    y: float

p1 = Point(1.0, 2.0)
p2 = Point(1.0, 2.0)
p3 = Point(3.0, 4.0)

print(p1)
print(p1 == p2)
print(p1 == p3)

Point(x=1.0, y=2.0)
True
False


### 10.1. Значения по умолчанию и методы.

In [ ]:
@dataclass
class Student:
    name: str
    age: int
    grades: list = None

    def __post_init__(self):
        if self.grades is None:
            self.grades = []

    def average(self):
        return sum(self.grades) / len(self.grades) if self.grades else 0

s = Student("Иван", 20, [4, 5, 5, 3])
print(s)
print(s.average())

Student(name='Иван', age=20, grades=[4, 5, 5, 3])
4.25


### 10.2. Параметры dataclass.

- `frozen=True` - объект становится неизменяемым (аналог `tuple`).
- `order=True` - добавляет методы сравнения `<`, `>`, `<=`, `>=`.
- `eq=False` - отключает авто-генерацию `__eq__`.

In [ ]:
@dataclass(frozen=True, order=True)
class Version:
    major: int
    minor: int

v1 = Version(1, 0)
v2 = Version(2, 1)

print(v1 < v2)
print(sorted([v2, v1]))

try:
    v1.major = 5
except Exception as err:
    print("Ошибка:", type(err).__name__)

True
[Version(major=1, minor=0), Version(major=2, minor=1)]
Ошибка: FrozenInstanceError


## 11. Магические методы.

Магические методы (dunder-методы, от *double underscore*) - это специальные методы, которые Python вызывает автоматически при использовании операторов, встроенных функций и некоторых синтаксических конструкций. Они позволяют объектам вашего класса вести себя как встроенные типы.

Название всегда обрамлено двумя подчёркиваниями: `__init__`, `__str__`, `__add__` и т.д.

### 11.1. Строковое представление: `__str__` и `__repr__`.

- `__str__` - "человекочитаемое" представление. Вызывается функциями `print()` и `str()`.
- `__repr__` - "отладочное" представление. Вызывается функцией `repr()`, а также когда объект выводится в консоли / ipynb ноутбуке.

Если `__str__` не определён, Python использует `__repr__`. Наоборот — не работает.

In [ ]:
class Book:
    def __init__(self, title, author, year):
        self.title = title
        self.author = author
        self.year = year

    def __str__(self):
        return f"«{self.title}» — {self.author} ({self.year})"

    def __repr__(self):
        return f"Book(title={self.title!r}, author={self.author!r}, year={self.year})"

b = Book("Властелин колец", "Толкин", 1954)
print(b)  # вызовет __str__
print(str(b))
print(repr(b))  # вызовет __repr__
b  # в ноутбуке отображается через __repr__

«Властелин колец» — Толкин (1954)
«Властелин колец» — Толкин (1954)
Book(title='Властелин колец', author='Толкин', year=1954)


Book(title='Властелин колец', author='Толкин', year=1954)

### 11.2. Арифметические операторы.

Для каждой арифметической операции есть свой магический метод:

| Оператор | Метод | Оператор | Метод |
|---|---|---|---|
| `+` | `__add__` | `+=` | `__iadd__` |
| `-` | `__sub__` | `-=` | `__isub__` |
| `*` | `__mul__` | `*=` | `__imul__` |
| `/` | `__truediv__` | `/=` | `__itruediv__` |
| `//` | `__floordiv__` | `//=` | `__ifloordiv__` |
| `%` | `__mod__` | `%=` | `__imod__` |
| `**` | `__pow__` | `**=` | `__ipow__` |

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __str__(self):
        return f"Vector({self.x}, {self.y})"

    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)

    def __sub__(self, other):
        return Vector(self.x - other.x, self.y - other.y)

    def __mul__(self, scalar):
        return Vector(self.x * scalar, self.y * scalar)

    def __truediv__(self, scalar):
        return Vector(self.x / scalar, self.y / scalar)

v1 = Vector(1, 2)
v2 = Vector(3, 4)

print(v1 + v2)
print(v1 - v2)
print(v1 * 3)
print(v1 / 2)

Vector(4, 6)
Vector(-2, -2)
Vector(3, 6)
Vector(0.5, 1.0)


### 11.3. "Правые" версии операторов: `__radd__`, `__rmul__` и т.д.

Что делать, если слева от оператора стоит не ваш объект? Например: `2 * vector`. Python сначала вызовет `int.__mul__(2, vector)` - но `int` не знает, как умножать на `Vector`, и вернёт `NotImplemented`. Тогда Python попробует "правую" версию - `Vector.__rmul__(vector, 2)`.

Правило: для каждой операции `__op__` существует зеркальная `__rop__`.

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __str__(self):
        return f"Vector({self.x}, {self.y})"

    def __mul__(self, scalar):
        print("Вызван __mul__")
        return Vector(self.x * scalar, self.y * scalar)

    def __rmul__(self, scalar):
        print("Вызван __rmul__")
        return Vector(self.x * scalar, self.y * scalar)

v = Vector(1, 2)
print(v * 3)  # слева Vector -> __mul__
print(3 * v)  # слева int -> Python пробует int.__mul__, не получается -> __rmul__

Вызван __mul__
Vector(3, 6)
Вызван __rmul__
Vector(3, 6)


### 11.4. Операторы сравнения.

| Оператор | Метод |
|---|---|
| `==` | `__eq__` |
| `!=` | `__ne__` |
| `<` | `__lt__` |
| `<=` | `__le__` |
| `>` | `__gt__` |
| `>=` | `__ge__` |

Если определить `__eq__`, Python перестаёт использовать сравнение по умолчанию (по идентичности). Также при определении `__eq__` стоит определить `__hash__`, если объект будет использоваться в множествах или как ключ словаря.

In [ ]:
class Money:
    def __init__(self, amount, currency="RUB"):
        self.amount = amount
        self.currency = currency

    def __str__(self):
        return f"{self.amount} {self.currency}"

    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return self.amount == other.amount and self.currency == other.currency

    def __lt__(self, other):
        if self.currency != other.currency:
            raise ValueError("Нельзя сравнивать разные валюты")
        return self.amount < other.amount

m1 = Money(100)
m2 = Money(100)
m3 = Money(200)

print(m1 == m2)
print(m1 == m3)
print(m1 < m3)
print(sorted([m3, m1, m2]))

True
False
True
[<__main__.Money object at 0x78f98ec938c0>, <__main__.Money object at 0x78f98edff110>, <__main__.Money object at 0x78f98edff610>]


### 11.5. Контейнерные методы: `__len__`, `__getitem__`, `__setitem__`, `__contains__`, `__iter__`.

Эти методы позволяют объекту вести себя как список или словарь:

- `__len__` - работает с `len(obj)`.
- `__getitem__` - доступ по индексу / ключу: `obj[i]`.
- `__setitem__` - присваивание: `obj[i] = value`.
- `__delitem__` - удаление: `del obj[i]`.
- `__contains__` - оператор `in`.
- `__iter__` - цикл `for x in obj`.

In [ ]:
class Playlist:
    def __init__(self):
        self._tracks = []

    def add(self, track):
        self._tracks.append(track)

    def __len__(self):
        return len(self._tracks)

    def __getitem__(self, index):
        return self._tracks[index]

    def __contains__(self, track):
        return track in self._tracks

    def __iter__(self):
        return iter(self._tracks)

p = Playlist()
p.add("Song A")
p.add("Song B")
p.add("Song C")

print(len(p))
print(p[0])
print("Song B" in p)

for track in p:
    print(track)

3
Song A
True
Song A
Song B
Song C


### 11.6. Контекстные менеджеры: `__enter__` и `__exit__`.

Позволяют использовать объект в конструкции `with`. Метод `__enter__` вызывается при входе в блок, `__exit__` - при выходе (в том числе при исключении).

In [ ]:
class Timer:
    def __enter__(self):
        import time
        self.start = time.time()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        import time
        self.end = time.time()
        print(f"Прошло: {self.end - self.start:.4f} секунд")
        return False  # не подавляем исключения

with Timer() as t:
    total = sum(range(1_000_000))
    print("Сумма:", total)

Сумма: 499999500000
Прошло: 0.0604 секунд


### 11.7. Вызываемые объекты: `__call__`.

Если определить `__call__`, экземпляр класса можно вызывать как функцию: `obj(args)`.

In [ ]:
class Multiplier:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, x):
        return x * self.factor

double = Multiplier(2)
triple = Multiplier(3)

print(double(5))
print(triple(5))
print(list(map(double, [1, 2, 3, 4])))

10
15
[2, 4, 6, 8]


### 11.8. Шпаргалка по магическим методам.

| Категория | Методы |
|---|---|
| Создание/уничтожение | `__init__`, `__new__`, `__del__` |
| Строковое представление | `__str__`, `__repr__`, `__format__` |
| Арифметика | `__add__`, `__sub__`, `__mul__`, `__truediv__`, `__floordiv__`, `__mod__`, `__pow__` |
| Правая арифметика | `__radd__`, `__rsub__`, `__rmul__`, `__rtruediv__`, ... |
| In-place арифметика | `__iadd__`, `__isub__`, `__imul__`, ... |
| Сравнение | `__eq__`, `__ne__`, `__lt__`, `__le__`, `__gt__`, `__ge__`, `__hash__` |
| Контейнеры | `__len__`, `__getitem__`, `__setitem__`, `__delitem__`, `__contains__`, `__iter__`, `__next__` |
| Контекст | `__enter__`, `__exit__` |
| Вызов | `__call__` |
| Преобразование | `__int__`, `__float__`, `__bool__`, `__index__` |

Полный список - в документации Python: *[Data model](https://docs.python.org/3/reference/datamodel.html)*.

## 12. Сравнение подходов.

| Подход | Когда использовать |
|---|---|
| Обычный класс с `__init__` | Есть логика, сложные методы, изменяемое состояние |
| `@dataclass` | Хранилище данных, минимум логики, много атрибутов |
| Абстрактный класс | Нужен строгий интерфейс для наследников |
| `@property` | Нужно вычисляемое значение как атрибут или контроль записи |
| `@classmethod` | Альтернативный конструктор, фабрика |
| `@staticmethod` | Утилита, логически привязанная к классу |
| Магические методы | Хочется, чтобы объект вёл себя как встроенный тип |

## Задачи.

### Задача 2. Библиотека.

Создайте класс `Library`, который хранит список книг (в виде объектов `Book` из прошлого занятия или в виде строк). Реализуйте:

- `@classmethod from_list(cls, titles)` - создаёт библиотеку из списка названий;
- `@staticmethod is_valid_title(title)` - возвращает `True`, если название не пустое и не длиннее 101 символов;
- `@property total` - возвращает количество книг;
- `__str__` - красивое строковое представление: `"Библиотека, книг: N"`.

In [ ]:
# Задача 1. Библиотека.

class Library:
    def __init__(self, titles):
        self.books = list(titles)

    @classmethod
    def from_list(cls, titles):
        return cls(titles)

    @staticmethod
    def is_valid_title(title):
        return 0 < len(title) <= 101

    @property
    def total(self):
        return len(self.books)

    def __str__(self):
        return f"Библиотека, книг: {self.total}"

Библиотека, книг: 2
2


### Задача 3. Вектор.

Создайте класс `Vector` (или `@dataclass`) с полями `x` и `y` типа `float`. Реализуйте:

- метод `length()` - длина вектора;
- `__add__` - сложение двух векторов;
- `__sub__` - вычитание;
- `__mul__` и `__rmul__` - умножение на число с обеих сторон;
- `__eq__` - сравнение по координатам;
- `__str__` и `__repr__`.

In [ ]:
# Задача 2. Вектор.

class Vector:
    def __init__(self, x, y):
        self.x = float(x)
        self.y = float(y)

    def length(self):
        return np.sqrt(self.x ** 2 + self.y ** 2)

    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)

    def __sub__(self, other):
        return Vector(self.x - other.x, self.y - other.y)

    def __mul__(self, number):
        return Vector(self.x * number, self.y * number)

    def __rmul__(self, number):
        return self * number

    def __eq__(self, other):
        return isinstance(other, Vector) and self.x == other.x and self.y == other.y

    def __str__(self):
        return f"({self.x}, {self.y})"

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

Vector(4, 6)
Vector(-2, -2)
Vector(3, 6)
Vector(2, 4)
2.23606797749979
True


### Задача 3. Дробь.

Создайте класс `Fraction` с полями `numerator` и `denominator`. Реализуйте:

- `__add__`, `__sub__`, `__mul__`, `__truediv__` - арифметика дробей;
- `__eq__`, `__lt__` - сравнение;
- `__str__` - вывод в виде `"3/4"`;
- сокращение дроби в `__init__` (через `math.gcd`).

In [ ]:
# Задача 3. Дробь.
from math import gcd

class Fraction:
    def __init__(self, numerator, denominator):
        if denominator == 0:
            raise ZeroDivisionError
        d = gcd(numerator, denominator)
        self.numerator = numerator // d
        self.denominator = denominator // d
        if self.denominator < 0:
            self.numerator *= -1
            self.denominator *= -1

    def __add__(self, other):
        return Fraction(self.numerator * other.denominator + other.numerator * self.denominator,
                        self.denominator * other.denominator)

    def __sub__(self, other):
        return Fraction(self.numerator * other.denominator - other.numerator * self.denominator,
                        self.denominator * other.denominator)

    def __mul__(self, other):
        return Fraction(self.numerator * other.numerator,
                        self.denominator * other.denominator)

    def __truediv__(self, other):
        return Fraction(self.numerator * other.denominator,
                        self.denominator * other.numerator)

    def __eq__(self, other):
        return self.numerator == other.numerator and self.denominator == other.denominator

    def __lt__(self, other):
        return self.numerator * other.denominator < other.numerator * self.denominator

    def __str__(self):
        return f"{self.numerator}/{self.denominator}"

5/6
1/6
1/6
3/2
False
True


### Задача 5. Сотрудники.

Создайте абстрактный класс `Employee` с абстрактными методами `salary()` и `position()`. Реализуйте наследников:

- `Manager` - фиксированная зарплата + бонус;
- `Developer` - почасовая ставка × количество часов.

Создайте `@classmethod from_string(cls, data)` у каждого класса, который парсит строку вида `"Иван;50000"`.

Добавьте `__str__`, выводящий имя, должность и зарплату.

In [ ]:
# Задача 4. Сотрудники.
from abc import ABC, abstractmethod

class Employee(ABC):
    def __init__(self, name):
        self.name = name

    @abstractmethod
    def salary(self):
        pass

    @abstractmethod
    def position(self):
        pass

    def __str__(self):
        return f"{self.name}, {self.position()}, зарплата: {self.salary()}"


class Manager(Employee):
    def __init__(self, name, fixed_salary, bonus=0):
        super().__init__(name)
        self.fixed_salary = fixed_salary
        self.bonus = bonus

    def salary(self):
        return self.fixed_salary + self.bonus

    def position(self):
        return "Менеджер"

    @classmethod
    def from_string(cls, data):
        parts = data.split(";")
        return cls(parts[0], float(parts[1]), float(parts[2]) if len(parts) > 2 else 0)


class Developer(Employee):
    def __init__(self, name, hourly_rate, hours=0):
        super().__init__(name)
        self.hourly_rate = hourly_rate
        self.hours = hours

    def salary(self):
        return self.hourly_rate * self.hours

    def position(self):
        return "Разработчик"

    @classmethod
    def from_string(cls, data):
        parts = data.split(";")
        return cls(parts[0], float(parts[1]), float(parts[2]) if len(parts) > 2 else 0)

Иван — Менеджер, зарплата: 60000.0
Мария — Разработчик, зарплата: 80000.0


### Задача 5. Аккаунт с историей.

Реализуйте класс `BankAccount`:

- приватный атрибут `__balance`;
- `@property balance` только для чтения;
- метод `deposit(amount)` - увеличивает баланс, кидает `ValueError` при неверной сумме;
- метод `withdraw(amount)` - уменьшает баланс с проверкой;
- список `_history` для хранения операций (используйте `@property` для чтения);
- `__str__` - выводит владельца и баланс;
- `__len__` - количество операций в истории.

In [ ]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.__balance = balance
        self._history = []

    @property
    def balance(self):
        return self.__balance

    @property
    def history(self):
        return self._history.copy()

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError
        self.__balance += amount
        self._history.append(f"Пополнение: {amount}")

    def withdraw(self, amount):
        if amount <= 0 or amount > self.__balance:
            raise ValueError
        self.__balance -= amount
        self._history.append(f"Снятие: {amount}")

    def __str__(self):
        return f"{self.owner}, баланс: {self.__balance}"

    def __len__(self):
        return len(self._history)

Счёт Иван: 1300 руб.
2
[('deposit', 500), ('withdraw', 200)]
1300


### Задача 7. dataclass-конфиг.

Создайте `@dataclass` `ServerConfig` с полями `host: str`, `port: int = 8080`, `timeout: float = 30.0`, `debug: bool = False`. Добавьте:

- метод `url()` - возвращает строку вида `http://host:port`;
- `@classmethod from_dict(cls, data)`, который создаёт объект из словаря.

Проверьте, что два объекта с одинаковыми значениями считаются равными.

In [ ]:
# Задача 6. dataclass-конфиг.
from dataclasses import dataclass

@dataclass
class ServerConfig:
    host: str
    port: int = 8080
    timeout: float = 30.0
    debug: bool = False

    def url(self):
        return f"http://{self.host}:{self.port}"

    @classmethod
    def from_dict(cls, data):
        return cls(**data)


config1 = ServerConfig("localhost")
config2 = ServerConfig.from_dict({"host": "localhost"})

print(config1.url())
print(config1 == config2)

http://localhost:8080
http://example.com:443
True
False


### Задача 7. Композиция и полиморфизм.

Создайте класс `Playlist`, который содержит список объектов `Track` (dataclass с полями `title`, `artist`, `duration`). Реализуйте:

- `total_duration()` - общая длительность;
- `add(track)` - добавление трека;
- `__len__` - количество треков;
- `__iter__` - перебор треков;
- `__getitem__` - доступ по индексу;
- `__contains__` - проверка наличия трека по названию;
- `__str__`.

In [ ]:
# Задача 7. Композиция и полиморфизм.
from dataclasses import dataclass
import numpy as np

@dataclass
class Track:
    title: str
    artist: str
    duration: float

class Playlist:
    def __init__(self):
        self.tracks = []

    def total_duration(self):
        return np.sum([track.duration for track in self.tracks])

    def add(self, track):
        self.tracks.append(track)

    def __len__(self):
        return len(self.tracks)

    def __iter__(self):
        return iter(self.tracks)

    def __getitem__(self, index):
        return self.tracks[index]

    def __contains__(self, title):
        return any(track.title == title for track in self.tracks)

    def __str__(self):
        return f"Плейлист: {len(self)} треков, длительность: {self.total_duration()}"

Плейлист «Мой плейлист», треков: 3, длительность: 635.0 сек.
3
Track(title='Song A', artist='Artist 1', duration=210)
True
False
Song A
Song B
Song C


### Задача 8. Матрица.

Создайте класс `Matrix` для квадратной матрицы 2×2. Реализуйте:

- `__add__`, `__sub__`, `__mul__` (умножение матриц);
- `__rmul__` для умножения на скаляр слева;
- `__eq__`, `__str__`, `__repr__`;
- `determinant()` - определитель.

In [ ]:
import numpy as np

class Matrix:
    def __init__(self, data):
        self.data = np.array(data)

    def __add__(self, other):
        return Matrix(self.data + other.data)

    def __sub__(self, other):
        return Matrix(self.data - other.data)

    def __mul__(self, other):
        return Matrix(self.data @ other.data)

    def __rmul__(self, scalar):
        return Matrix(scalar * self.data)

    def __eq__(self, other):
        return np.array_equal(self.data, other.data)

    def __str__(self):
        return str(self.data)

    def __repr__(self):
        return f"Matrix({self.data.tolist()})"

    def determinant(self):
        return np.linalg.det(self.data)

In [ ]:
# Задача 8. Матрица.


[[6, 8], [10, 12]]
[[-4, -4], [-4, -4]]
[[19, 22], [43, 50]]
[[2, 4], [6, 8]]
-2
True
